In [0]:
# ============================================================
# 01_bronze_ingest_Auto_load_4 — Bronze ingestion via Auto Loader
# Author: oakville3456
# Updated: 2026-06-06
# Branch: main
# Purpose: Incrementally ingest CSV files from ADLS raw-landing
#          into Bronze Delta table via Unity Catalog
# ============================================================

from pyspark.sql import functions as F
from datetime import datetime, timezone

# ─────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────
RAW         = "abfss://raw-landing@saretailsalesdev.dfs.core.windows.net/sales/"
BRONZE_TBL  = "adb_retail_dev.bronze.sales"
CHECKPOINT  = "abfss://bronze@saretailsalesdev.dfs.core.windows.net/_checkpoint/bronze_sales"
SCHEMA_LOC  = "abfss://bronze@saretailsalesdev.dfs.core.windows.net/_schema/bronze_sales"
RUN_LOG_TBL = "adb_retail_dev.bronze.run_log_sales"

run_start = datetime.now(timezone.utc)

# ─────────────────────────────────────────────────────────────
# SNAPSHOT: Delta version BEFORE run
# ─────────────────────────────────────────────────────────────
try:
    version_before = (
        spark.sql(f"DESCRIBE HISTORY {BRONZE_TBL} LIMIT 1")
        .collect()[0]
        .version
    )
except Exception:
    # Table does not exist yet (first run)
    version_before = -1

# ─────────────────────────────────────────────────────────────
# AUTO LOADER READ
# Only new files since last checkpoint are processed
# ─────────────────────────────────────────────────────────────
raw = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", SCHEMA_LOC)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("header", "true")
        .load(RAW)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
)

# ─────────────────────────────────────────────────────────────
# WRITE TO BRONZE (Unity Catalog managed table)
# ─────────────────────────────────────────────────────────────
query = (
    raw.writeStream
        .format("delta")
        .option("checkpointLocation", CHECKPOINT)
        .option("mergeSchema", "true")
        .trigger(availableNow=True)     # process all pending files, then stop
        .toTable(BRONZE_TBL)            # writes to Unity Catalog table
)

query.awaitTermination()
query.stop()

# ─────────────────────────────────────────────────────────────
# SNAPSHOT: Delta version AFTER run
# ─────────────────────────────────────────────────────────────
version_after = (
    spark.sql(f"DESCRIBE HISTORY {BRONZE_TBL} LIMIT 1")
    .collect()[0]
    .version
)

versions_added = version_after - version_before

# ─────────────────────────────────────────────────────────────
# METRICS FROM DELTA LOG (no expensive full table scan)
# ─────────────────────────────────────────────────────────────
history = spark.sql(f"DESCRIBE HISTORY {BRONZE_TBL}")
recent_commits = history.filter(F.col("version") > version_before)

rows_added = (
    recent_commits
    .select(F.col("operationMetrics.numOutputRows").cast("bigint").alias("rows"))
    .agg(F.sum("rows"))
    .collect()[0][0]
)

files_added = (
    recent_commits
    .select(F.col("operationMetrics.numFiles").cast("bigint").alias("files"))
    .agg(F.sum("files"))
    .collect()[0][0]
)

# Handle nulls if no new commits or metrics missing
rows_added  = rows_added  or 0
files_added = files_added or 0

# ─────────────────────────────────────────────────────────────
# RUN LOG — append to Unity Catalog table
# ─────────────────────────────────────────────────────────────
log = spark.createDataFrame([{
    "run_timestamp":  run_start,
    "version_before": int(version_before),
    "version_after":  int(version_after),
    "versions_added": int(versions_added),
    "rows_added":     int(rows_added),
    "files_added":    int(files_added),
    "status":         "new_data" if rows_added > 0 else "no_new_data"
}])

(
    log.write
       .format("delta")
       .mode("append")
       .option("mergeSchema", "true")
       .saveAsTable(RUN_LOG_TBL)
)

# ─────────────────────────────────────────────────────────────
# SUMMARY
# ─────────────────────────────────────────────────────────────
print("────────────────────────────────────────────")
print(f" Bronze ingestion completed at {run_start}")
print(f" Versions added : {versions_added}")
print(f" Rows added     : {rows_added}")
print(f" Files added    : {files_added}")
print(f" Status         : {'NEW DATA INGESTED' if rows_added > 0 else 'NO NEW DATA'}")
print("────────────────────────────────────────────")